In [ ]:
# Cell 0 · Install & Import
!pip install awswrangler matplotlib --quiet
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('OK')

In [ ]:
# Cell 1 · Config
S3_BUCKET   = 'gold-lstm-forecast'
SILVER_PATH = f's3://{S3_BUCKET}/silver/xauusd_daily_clean.parquet'
GOLD_PATH   = f's3://{S3_BUCKET}/gold/xauusd_daily/features/xauusd_features.parquet'
print(f'Silver : {SILVER_PATH}')
print(f'Gold   : {GOLD_PATH}')

In [ ]:
# Cell 2 · Load Silver Data
df = wr.s3.read_parquet(path=SILVER_PATH)
df = df.sort_values('date').reset_index(drop=True)
df['date'] = pd.to_datetime(df['date'])
print(f'Shape      : {df.shape}')
print(f'Date range : {df["date"].min().date()} -> {df["date"].max().date()}')

In [ ]:
# Cell 3 · Feature Engineering
df['return']       = df['close'].pct_change()
df['ma7']          = df['close'].rolling(7).mean()
df['ma14']         = df['close'].rolling(14).mean()
df['ma30']         = df['close'].rolling(30).mean()
df['ma60']         = df['close'].rolling(60).mean()
df['volatility_7'] = df['return'].rolling(7).std()
df['momentum_7']   = df['close'] - df['close'].shift(7)
df['target']       = df['close'].shift(-1)

print(f'  Shape before dropna : {df.shape}')
df = df.dropna().reset_index(drop=True)
print(f'  Shape after dropna  : {df.shape}')
print(f'  Dropped rows        : 61 (60 from ma60 window + 1 from target shift)')
print(f'  Date range          : {df["date"].min().date()} -> {df["date"].max().date()}')

In [ ]:
# Cell 4 · Feature Sample & Stats
features = ['close','return','ma7','ma14','ma30','ma60','volatility_7','momentum_7']

print('=== Feature Sample ===')
print(df[['date'] + features + ['target']].head(5).to_string(index=False))
print()
print('=== Feature Stats ===')
print(df[features].describe().round(4).to_string())

In [ ]:
# Cell 5 · Dataset Schema
cols   = ['date','close','return','ma7','ma14','ma30','ma60','volatility_7','momentum_7','target']
sample = df[cols].head(5).copy()
sample['date'] = sample['date'].astype(str).str[:10]
for c in cols[1:]:
    sample[c] = sample[c].round(4)

desc = {
    'date'        : ('datetime64',  'Trading date'),
    'close'       : ('float64',     'Closing price (USD)'),
    'return'      : ('float64',     'Daily % change from previous day'),
    'ma7'         : ('float64',     '7-day moving average'),
    'ma14'        : ('float64',     '14-day moving average'),
    'ma30'        : ('float64',     '30-day moving average'),
    'ma60'        : ('float64',     '60-day moving average'),
    'volatility_7': ('float64',     'Rolling std of return (7-day)'),
    'momentum_7'  : ('float64',     'Close - Close 7 days ago'),
    'target'      : ('float64',     'Next-day closing price (label)'),
}

print('='*110)
print(f'{"GOLD PRICE DATASET — Feature Engineering Sample (Gold Layer)":^110}')
print('='*110)
print(f'{"Column":<16} {"Type":<12} {"Description":<45} {"Sample Value 1":>18} {"Sample Value 2":>15}')
print('-'*110)
for col in cols:
    dtype, description = desc[col]
    v1 = str(sample[col].iloc[0])
    v2 = str(sample[col].iloc[1])
    print(f'{col:<16} {dtype:<12} {description:<45} {v1:>18} {v2:>15}')
print('='*110)
print(f'  Total rows : {len(df):,}  |  Date range : {df["date"].min().date()} -> {df["date"].max().date()}  |  Features : {len(cols)-2}  |  Target : next-day close')
print('='*110)

In [ ]:
# Cell 6 · EDA Visualization
fig, axes = plt.subplots(4, 1, figsize=(14, 14))
fig.suptitle('Feature Engineering Overview', fontsize=13, fontweight='bold')

axes[0].plot(df['date'], df['close'],  color='#E8A020', lw=0.8, label='Close')
axes[0].plot(df['date'], df['ma7'],    color='#4A90D9', lw=1.0, label='MA7',  alpha=0.8)
axes[0].plot(df['date'], df['ma30'],   color='#E74C3C', lw=1.0, label='MA30', alpha=0.8)
axes[0].plot(df['date'], df['ma60'],   color='#2ECC71', lw=1.0, label='MA60', alpha=0.8)
axes[0].set_title('Close Price with Moving Averages')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['date'], df['return']*100, color='#4A90D9', lw=0.6, alpha=0.8)
axes[1].axhline(0, color='black', lw=0.8, linestyle='--')
axes[1].set_title('Daily Return (%)')
axes[1].set_ylabel('%')
axes[1].grid(True, alpha=0.3)

axes[2].plot(df['date'], df['volatility_7']*100, color='#E74C3C', lw=0.8)
axes[2].set_title('Rolling Volatility 7-day')
axes[2].set_ylabel('%')
axes[2].grid(True, alpha=0.3)

axes[3].plot(df['date'], df['momentum_7'], color='#2ECC71', lw=0.8)
axes[3].axhline(0, color='black', lw=0.8, linestyle='--')
axes[3].set_title('Momentum 7-day')
axes[3].set_ylabel('USD')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('03_feature_engineering.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved -> 03_feature_engineering.png')

In [ ]:
# Cell 7 · Save to Gold Layer
wr.s3.to_parquet(df=df, path=GOLD_PATH, index=False)
print(f'Gold saved -> {GOLD_PATH}')
print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
# Cell 8 · Verify Read-back
verify = wr.s3.read_parquet(path=GOLD_PATH)
print('=== Verify Gold ===')
print(f'  Shape      : {verify.shape}')
print(f'  Date range : {verify["date"].min()} -> {verify["date"].max()}')
print(f'  Missing    : {verify.isnull().sum().sum()}')
print()
print('Notebook 03 DONE')
print('Next -> 04_Preprocessing.ipynb')